# Sudoku-12 : Résolution avec Z3 SMT Solver (C#)

**Navigation** : [<< Sudoku-11 Choco C#](Sudoku-11-Choco-Csharp.ipynb) | [Index](README.md) | [Sudoku-13 Symbolic Automata C# >>](Sudoku-13-SymbolicAutomata-Csharp.ipynb)

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :

1. **Comprendre les principes de base de Z3** - SMT solver, satisfiabilité, théories et contraintes
2. **Modéliser un problème de contraintes avec Z3** - Représentation des variables, construction des contraintes, résolution
3. **Comparaison des approches de résolution** - Entiers vs vecteurs de bits, API de substitution et tactiques

### Prerequis

- Avoir suivi le notebook **[Sudoku-10 OR-Tools](Sudoku-10-ORTools-Csharp.ipynb)** (recommandé pour comprendre les concepts de contraintes)
- Notions de base en programmation logique et en raisonnement symbolique

### Durée estimée : ~40 minutes

### Voir aussi

- [CSP-3-Avance](../Search/Part2-CSP/CSP-3-Advanced.ipynb) - Contraintes globales avancees
- [SymbolicAI](../SymbolicAI/README.md) - Série complète sur l'IA symbolique avec Z3

## Introduction à Z3

Z3 est un SMT solver (Satisfiability Modulo Theories) qui peut être utilisé pour résoudre des problèmes de logique du premier ordre. Il permet de vérifier la satisfiabilité d'une formule logique sous certaines contraintes et est particulièrement efficace pour résoudre des problèmes impliquant des contraintes complexes, comme les puzzles de Sudoku.

Un SMT solver combine des techniques de résolution SAT (Satisfiability) avec des théories comme l'arithmétique, les tableaux, les bit-vectors, etc., permettant ainsi de résoudre des problèmes logiques beaucoup plus riches.

**Références :**
- [Exemples en C#](https://github.com/Z3Prover/z3/blob/master/examples/dotnet/Program.cs)
- [Programming Z3](https://theory.stanford.edu/~nikolaj/programmingz3.html)
- [Z3Py Guide Examples](https://ericpony.github.io/z3py-tutorial/guide-examples.htm) en Python

## Configuration de l'environnement

In [1]:
#r "nuget: Microsoft.Z3"

Installed Packages Microsoft.Z3, 4.12.2

## Importation des Classes de Base

Nous allons importer les classes de base définies dans le notebook précédent, fournissant notamment la représentation, le chargement et l'affichage de Sudokus, et l'infrastructure de résolution.


In [2]:
#!import Sudoku-0-Environment-Csharp.ipynb

# Sudoku-0 : Environnement et Classes de Base (C#)

**Navigation** : [Index](README.md) | [Sudoku-1 Backtracking C# >>](Sudoku-1-Backtracking-Csharp.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Comprendre la structure de donnees `SudokuGrid` et ses methodes principales
2. Utiliser `ISudokuSolver` pour implementer un solveur de Sudoku
3. Exploiter `SudokuHelper` pour charger des grilles et tester des solveurs
4. Comparer les performances de plusieurs solveurs sur differentes difficultes

**Prerequis** : Notions de base en C# (.NET Interactive)  
**Duree estimee** : ~15 min

Installed Packages Plotly.NET, 5.1.0

## Définition de la classe SudokuGrid

Nous définissons ici la classe SudokuGrid qui représente une grille de Sudoku et fournit des méthodes pour manipuler et afficher les grilles.


SudokuGrid defini.


### Interpretation : Structure de donnees pour la grille Sudoku

**Sortie obtenue** : La classe `SudokuGrid` encapsule toutes les operations de manipulation, validation et affichage d'une grille de Sudoku 9x9.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| `Cells[9,9]` | int[,] | Stockage interne des valeurs (0 = vide) |
| `AllNeighbours` | 27 x 9 positions | Pre-calcul des voisins ligne/colonne/bloc |
| `CellNeighbours[9][9]` | ~20 positions chacune | Voisins directs de chaque cellule |
| `GetAvailableNumbers()` | int[] | Candidats valides pour une cellule |
| `NbErrors()` | int | Nombre de conflits + modifications erronees |

**Points cles** :
1. **Pre-calcul des voisins** : `AllNeighbours` et `CellNeighbours` sont calcules une seule fois a l'initialisation, evitant les recalculs couteux
2. **Conversion flexible** : Methodes pour convertir entre tableaux 1D, 2D et jagged arrays (utile pour differents formats de fichiers)
3. **Validation robuste** : `NbErrors` compte a la fois les doublons (ligne/colonne/bloc) et les modifications de indices pre-remplis
4. **Parsing tolerant** : `ReadMultiSudoku` accepte plusieurs formats (`.`, `X`, `-`, espaces)

> **Note technique** : La structure `CellNeighbours[i][j]` contient environ 20 positions (8 ligne + 8 colonne + 4 bloc, moins les doublons). Ce pre-calcul est crucial pour les performances des algorithmes de backtracking et de propagation de contraintes.

...

SudokuHelper defini.


### Interpretation : Infrastructure de test et benchmark

**Sortie obtenue** : La classe `SudokuHelper` fournit une infrastructure complete pour tester et comparer les solveurs de Sudoku.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| `GetSudokus()` | 51/95/100 grilles | Trois niveaux de difficulte (Easy/Medium/Hard) |
| `TestSolvers()` | Performance multi-solveurs | Execution parallele avec timeout |
| `DisplayResults()` | Graphiques Plotly.NET | Visualisation des temps de resolution |
| `SolveSudoku()` | Test unitaire | Resolution individuelle avec affichage |

**Points cles** :
1. **Chargement intelligent** : Recherche recursive du dossier `Puzzles` dans l'arborescence
2. **Robustesse** : Gestion des timeouts (3000ms par defaut) et exceptions
3. **Mesures** : Temps d'execution total + nombre de grilles resolues
4. **Disqualification** : Un solver echouant sur une grille est disqualifie pour la difficulte

> **Note technique** : La methode `TestSolvers` utilise `Interlocked.Increment` pour un thread-safe increment du compteur de solutions. Le `CancellationToken` permet d'interrompre proprement les solveurs trop lents.

## Exercice : Validation d'une grille Sudoku

### Enonce

Implementez une methode `IsValidSolution` qui verifie qu'une grille est une solution valide de Sudoku, c'est-a-dire que chaque ligne, chaque colonne et chaque bloc 3x3 contient exactement une fois chaque chiffre de 1 a 9.

Utilisez cette methode pour valider les resultats de `SudokuHelper.SolveSudoku`.

**Indices :**

- Parcourez les 9 lignes, 9 colonnes et 9 blocs
- Pour chaque unite, verifiez que les 9 chiffres sont tous presents sans doublon
- `SudokuGrid.AllNeighbours` contient deja les indices des unites

Exercice a completer


## Resume et perspectives

Ce notebook a pose les fondations de toute la serie Sudoku en definissant trois composants essentiels. La classe `SudokuGrid` encapsule la representation d'une grille 9x9 avec le pre-calcul des voisins (`AllNeighbours`, `CellNeighbours`), ce qui evite les recalculs couteux lors de la resolution. L'interface `ISudokuSolver` implante le pattern Strategie, permettant de permuter les algorithmes de resolution sans modifier le code client. Enfin, la classe `SudokuHelper` fournit une infrastructure de benchmark complete avec chargement de puzzles, mesures de performance et visualisation Plotly.NET.

L'infrastructure de test (`TestSolvers`, `DisplayResults`) permet de comparer objectivement les solveurs sur trois niveaux de difficulte (Easy, Medium, Hard) avec gestion des timeouts et des disqualifications. Ce cadre de benchmark sera utilise dans tous les notebooks suivants pour mesurer les performances de chaque algorithme.

Le notebook suivant, [Sudoku-1-Backtracking](Sudoku-1-Backtracking-Csharp.ipynb), utilise ces classes pour implementer le premier algorithme de resolution : le backtracking recursif avec ses heuristiques d'amelioration.

### 1. Implémentation de base avec des entiers

Comme nous allons tester plusieurs stratégies de résolution, nous commencerons par implémenter un solver simple utilisant des entiers pour représenter les cellules du Sudoku et fournissant les méthodes pour construire les contraintes.


In [3]:
// Importer les bibliothèques nécessaires
using System.Diagnostics;
using System;
using System.Collections;
using System.Collections.Generic;
using Microsoft.Z3;

// Classe pour résoudre le Sudoku en utilisant Z3 avec des entiers
public class Z3IntSolverSimple : ISudokuSolver
{
    public static Context ctx = new Context();
    public static BoolExpr _GenericContraints;
    public static IntExpr[][] CellVariables = new IntExpr[9][];
    public Z3IntSolverSimple()
    {
        // Initialiser les variables de cellule
        for (uint i = 0; i < 9; i++)
        {
            CellVariables[i] = new IntExpr[9];
            for (uint j = 0; j < 9; j++)
                CellVariables[i][j] = (IntExpr)ctx.MkConst(ctx.MkSymbol("x_" + (i + 1) + "_" + (j + 1)), ctx.IntSort);
        }
    }

    // Contraintes génériques pour tous les Sudokus, conservées en mémoire pour éviter de les recalculer
    public static BoolExpr GenericContraints
    {
        get
        {
            if (_GenericContraints == null)
            {
                _GenericContraints = GetGenericConstraints();
            }
            return _GenericContraints;
        }
    }
    
    // Générer les contraintes génériques
    public static BoolExpr GetGenericConstraints()
    {
        // Chaque cellule contient une valeur entre 1 et 9
        Expr[][] cells_c = new Expr[9][];
        for (uint i = 0; i < 9; i++)
        {
            cells_c[i] = new BoolExpr[9];
            for (uint j = 0; j < 9; j++)
                cells_c[i][j] = ctx.MkAnd(ctx.MkLe(ctx.MkInt(1), CellVariables[i][j]),
                                          ctx.MkLe(CellVariables[i][j], ctx.MkInt(9)));
        }
        // Chaque ligne contient des chiffres distincts
        BoolExpr[] rows_c = new BoolExpr[9];
        for (uint i = 0; i < 9; i++)
            rows_c[i] = ctx.MkDistinct(CellVariables[i]);
        // Chaque colonne contient des chiffres distincts
        BoolExpr[] cols_c = new BoolExpr[9];
        for (uint j = 0; j < 9; j++)
        {
            IntExpr[] column = new IntExpr[9];
            for (uint i = 0; i < 9; i++)
                column[i] = CellVariables[i][j];
            cols_c[j] = ctx.MkDistinct(column);
        }
        // Chaque carré 3x3 contient des chiffres distincts
        BoolExpr[][] sq_c = new BoolExpr[3][];
        for (uint i0 = 0; i0 < 3; i0++)
        {
            sq_c[i0] = new BoolExpr[3];
            for (uint j0 = 0; j0 < 3; j0++)
            {
                IntExpr[] square = new IntExpr[9];
                for (uint i = 0; i < 3; i++)
                    for (uint j = 0; j < 3; j++)
                        square[3 * i + j] = CellVariables[3 * i0 + i][3 * j0 + j];
                sq_c[i0][j0] = ctx.MkDistinct(square);
            }
        }
        BoolExpr sudoku_c = ctx.MkTrue();
        foreach (BoolExpr[] t in cells_c)
            sudoku_c = ctx.MkAnd(ctx.MkAnd(t), sudoku_c);
        sudoku_c = ctx.MkAnd(ctx.MkAnd(rows_c), sudoku_c);
        sudoku_c = ctx.MkAnd(ctx.MkAnd(cols_c), sudoku_c);
        foreach (BoolExpr[] t in sq_c)
            sudoku_c = ctx.MkAnd(ctx.MkAnd(t), sudoku_c);
        return sudoku_c;
    }

    // Générer les contraintes spécifiques à une grille de Sudoku donnée
    public BoolExpr GetPuzzleConstraints(SudokuGrid grid)
    {
        BoolExpr instance_c = ctx.MkTrue();
        for (uint i = 0; i < 9; i++)
            for (uint j = 0; j < 9; j++)
                if (grid.Cells[i,j] != 0)
                {
                    instance_c = ctx.MkAnd(instance_c,
                        (BoolExpr)ctx.MkEq(CellVariables[i][j], ctx.MkInt(grid.Cells[i,j])));
                }
        return instance_c;
    }

    // Résoudre le Sudoku
    public SudokuGrid Solve(SudokuGrid grid)
    {
        SudokuGrid solution = new SudokuGrid();
        var sudoku_c = GenericContraints;
        var instance_c = GetPuzzleConstraints(grid);
        Solver s = ctx.MkSolver();
        s.Assert(sudoku_c);
        s.Assert(instance_c);        
        if (s.Check() == Status.SATISFIABLE)
        {
            Model m = s.Model;
            for (uint i = 0; i < 9; i++)
            {
                for (uint j = 0; j < 9; j++)
                {
                    solution.Cells[i,j] = ((IntNum)m.Evaluate(CellVariables[i][j])).Int;
                }
            }
        }
        else
        {
            Console.WriteLine("Failed to solve sudoku");
            throw new Exception("Failed to solve sudoku");
        }
        return solution;
    }
}
Console.WriteLine("Classe Z3IntSolverSimple definie (solveur Z3 avec entiers)");

Classe Z3IntSolverSimple definie (solveur Z3 avec entiers)


#### Démonstration : résoudre une vraie grille avec l'encodage entier

Avant de comparer les performances, vérifions que le solveur produit bien une grille valide. On charge une grille *facile*, on l'affiche (les `0` marquent les cases à remplir), puis on la résout avec `Z3IntSolverSimple`. La sortie montre la grille de départ, la grille complète, et le nombre d'erreurs restantes : `0` confirme une solution correcte.

In [4]:
// Demonstration : charger une grille réelle, la résoudre et comparer avant / après.
var puzzleDemo = SudokuHelper.GetSudokus(SudokuDifficulty.Easy)[0];
Console.WriteLine("Grille initiale (les 0 sont les cases a remplir) :");
Console.WriteLine(puzzleDemo);

var solvedInt = new Z3IntSolverSimple().Solve(puzzleDemo);
Console.WriteLine("Grille resolue par Z3IntSolverSimple :");
Console.WriteLine(solvedInt);
Console.WriteLine($"Nombre d'erreurs restantes (0 = grille valide) : {solvedInt.NbErrors(puzzleDemo)}");

Grille initiale (les 0 sont les cases a remplir) :


-------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------


Grille resolue par Z3IntSolverSimple :


-------------------------------
| 9  6  2 | 1  8  5 | 4  7  3 | 
| 1  7  4 | 9  6  3 | 8  2  5 | 
| 5  3  8 | 4  2  7 | 1  6  9 | 
-------------------------------
| 8  2  6 | 3  5  9 | 7  4  1 | 
| 3  5  7 | 8  1  4 | 2  9  6 | 
| 4  9  1 | 6  7  2 | 5  3  8 | 
-------------------------------
| 2  4  9 | 5  3  8 | 6  1  7 | 
| 7  1  5 | 2  9  6 | 3  8  4 | 
| 6  8  3 | 7  4  1 | 9  5  2 | 
-------------------------------


Nombre d'erreurs restantes (0 = grille valide) : 0


### 2. Utilisation de vecteurs de bits

Nous allons maintenant explorer une autre piste d'amélioration en utilisant des vecteurs de bits pour représenter les cellules du Sudoku. Cette approche peut réduire la taille des variables et améliorer les performances du solver.

Comme nous testerons plusieurs types de résolution, nous introduisons une classe de base qui fournit les contraintes.

In [5]:
using Microsoft.Z3;

public abstract class Z3BitVectorSolverBase : ISudokuSolver
{
    public static Context ctx = new Context();
    public static BoolExpr _GenericContraints;
    public static BitVecExpr[][] CellVariables = new BitVecExpr[9][];

    private static Sort BitVectorSort = ctx.MkBitVecSort(4);

    public static Solver _ReusableSolver;

    public Z3BitVectorSolverBase()
    {
        // Initialiser les variables de cellule en tant que vecteurs de 4 bits
        for (uint i = 0; i < 9; i++)
        {
            CellVariables[i] = new BitVecExpr[9];
            for (uint j = 0; j < 9; j++)
                CellVariables[i][j] = (BitVecExpr)ctx.MkConst(ctx.MkSymbol("x_" + (i + 1) + "_" + (j + 1)), BitVectorSort);
        }
    }

    // Contraintes génériques pour le Sudoku
    public static BoolExpr GenericContraints
    {
        get
        {
            if (_GenericContraints == null)
            {
                _GenericContraints = GetGenericConstraints();
            }
            return _GenericContraints;
        }
    }

    public static Solver ReusableSolver
    {
        get
        {
            if (_ReusableSolver == null)
            {
                _ReusableSolver = MakeReusableSolver();
            }
            return _ReusableSolver;
        }
    }

    public static Solver MakeReusableSolver()
    {
        Solver s = ctx.MkSolver();
        s.Assert(GenericContraints);
        return s;
    }

    // Obtenir l'expression constante pour une valeur donnée
    protected static BitVecExpr GetConstExpr(int value)
    {
        return (BitVecExpr)ctx.MkNumeral(value, BitVectorSort);
    }

    // Générer les contraintes génériques pour les vecteurs de bits
    public static BoolExpr GetGenericConstraints()
    {
        // Chaque cellule contient une valeur entre 1 et 9
        Expr[][] cells_c = new Expr[9][];
        for (uint i = 0; i < 9; i++)
        {
            cells_c[i] = new BoolExpr[9];
            for (uint j = 0; j < 9; j++)
                cells_c[i][j] = ctx.MkAnd(ctx.MkBVULE(GetConstExpr(1), CellVariables[i][j]),
                    ctx.MkBVULE(CellVariables[i][j], GetConstExpr(9)));
        }

        // Chaque ligne contient des chiffres distincts
        BoolExpr[] rows_c = new BoolExpr[9];
        for (uint i = 0; i < 9; i++)
            rows_c[i] = ctx.MkDistinct(CellVariables[i]);

        // Chaque colonne contient des chiffres distincts
        BoolExpr[] cols_c = new BoolExpr[9];
        for (uint j = 0; j < 9; j++)
        {
            BitVecExpr[] column = new BitVecExpr[9];
            for (uint i = 0; i < 9; i++)
                column[i] = CellVariables[i][j];

            cols_c[j] = ctx.MkDistinct(column);
        }

        // Chaque carré 3x3 contient des chiffres distinct
        BoolExpr[][] sq_c = new BoolExpr[3][];
        for (uint i0 = 0; i0 < 3; i0++)
        {
            sq_c[i0] = new BoolExpr[3];
            for (uint j0 = 0; j0 < 3; j0++)
            {
                BitVecExpr[] square = new BitVecExpr[9];
                for (uint i = 0; i < 3; i++)
                for (uint j = 0; j < 3; j++)
                    square[3 * i + j] = CellVariables[3 * i0 + i][3 * j0 + j];
                sq_c[i0][j0] = ctx.MkDistinct(square);
            }
        }

        BoolExpr sudoku_c = ctx.MkTrue();
        foreach (BoolExpr[] t in cells_c)
            sudoku_c = ctx.MkAnd(ctx.MkAnd(t), sudoku_c);
        sudoku_c = ctx.MkAnd(ctx.MkAnd(rows_c), sudoku_c);
        sudoku_c = ctx.MkAnd(ctx.MkAnd(cols_c), sudoku_c);
        foreach (BoolExpr[] t in sq_c)
            sudoku_c = ctx.MkAnd(ctx.MkAnd(t), sudoku_c);

        return sudoku_c;
    }

    // Générer les contraintes spécifiques à une grille de Sudoku donnée
    public BoolExpr GetPuzzleConstraints(SudokuGrid grid)
    {
        BoolExpr instance_c = ctx.MkTrue();
        for (uint i = 0; i < 9; i++)
        for (uint j = 0; j < 9; j++)
            if (grid.Cells[i,j] != 0)
            {
                instance_c = ctx.MkAnd(instance_c,
                    (BoolExpr)ctx.MkEq(CellVariables[i][j], GetConstExpr(grid.Cells[i,j])));
            }

        return instance_c;
    }

    // Méthode abstraite pour résoudre le Sudoku
    public abstract SudokuGrid Solve(SudokuGrid s);
}

Console.WriteLine("Classe Z3BitVectorSolverBase definie (solveur Z3 abstrait avec vecteurs de bits)");

Classe Z3BitVectorSolverBase definie (solveur Z3 abstrait avec vecteurs de bits)


### 3. Implémentation simple avec vecteurs de bits

Nous allons implémenter un solver simple en utilisant des vecteurs de bits.

## Exercice : Résoudre un Killer Sudoku avec contrainte de somme

**Objectif :**
Ajoutez une contrainte de somme sur un bloc 3x3 pour modéliser un Killer Sudoku.

**Indice :**
Utilisez MkAdd pour sommer les variables d'un bloc et MkEq pour fixer la somme attendue.


In [6]:
// EXERCICE : Résoudre un Killer Sudoku avec contrainte de somme
public int[,] SolveWithSumConstraint(int[,] puzzle, int blockRow, int blockCol, int targetSum)
{
    // TODO: Ajoutez une contrainte de somme sur le bloc (blockRow, blockCol)
    // et resolvez le puzzle
    return null; // TODO etudiant
}


In [7]:
using Microsoft.Z3;

public class Z3BitVectorSolverSimple : Z3BitVectorSolverBase
{
    public override SudokuGrid Solve(SudokuGrid s)
    {
        SudokuGrid solution = new SudokuGrid();
        SudokuSolve(s, ref solution);
        return solution;
    }

    public void SudokuSolve(SudokuGrid grid, ref SudokuGrid solution)
    {
        var sudoku_c = GenericContraints;
        var instance_c = GetPuzzleConstraints(grid);
        Solver s = ctx.MkSolver();
        s.Assert(sudoku_c);
        s.Assert(instance_c);

        if (s.Check() == Status.SATISFIABLE)
        {
            Model m = s.Model;
            for (uint i = 0; i < 9; i++)
            {
                for (uint j = 0; j < 9; j++)
                {
                    solution.Cells[i,j] = ((BitVecNum)m.Evaluate(CellVariables[i][j])).Int;
                }
            }
        }
        else
        {
            Console.WriteLine("Failed to solve sudoku");
            throw new Exception("Failed to solve sudoku");
        }
    }
}

Console.WriteLine("Classe Z3BitVectorSolverSimple definie (solveur Z3 BitVector simple)");

Classe Z3BitVectorSolverSimple definie (solveur Z3 BitVector simple)


#### Démonstration : la même grille avec l'encodage en vecteurs de bits

L'encodage précédent représente chaque case par un entier ; celui-ci la représente par un **vecteur de bits** de 4 bits. On résout la même grille facile pour vérifier que ce second encodage aboutit lui aussi à une solution valide.

In [8]:
// Demonstration : le même puzzle, résolu cette fois par l'encodage en vecteurs de bits.
var puzzleDemoBv = SudokuHelper.GetSudokus(SudokuDifficulty.Easy)[0];
Console.WriteLine("Grille initiale :");
Console.WriteLine(puzzleDemoBv);

var solvedBv = new Z3BitVectorSolverSimple().Solve(puzzleDemoBv);
Console.WriteLine("Grille resolue par Z3BitVectorSolverSimple :");
Console.WriteLine(solvedBv);
Console.WriteLine($"Nombre d'erreurs restantes (0 = grille valide) : {solvedBv.NbErrors(puzzleDemoBv)}");

Grille initiale :


-------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------


Grille resolue par Z3BitVectorSolverSimple :


-------------------------------
| 9  6  2 | 1  8  5 | 4  7  3 | 
| 1  7  4 | 9  6  3 | 8  2  5 | 
| 5  3  8 | 4  2  7 | 1  6  9 | 
-------------------------------
| 8  2  6 | 3  5  9 | 7  4  1 | 
| 3  5  7 | 8  1  4 | 2  9  6 | 
| 4  9  1 | 6  7  2 | 5  3  8 | 
-------------------------------
| 2  4  9 | 5  3  8 | 6  1  7 | 
| 7  1  5 | 2  9  6 | 3  8  4 | 
| 6  8  3 | 7  4  1 | 9  5  2 | 
-------------------------------


Nombre d'erreurs restantes (0 = grille valide) : 0


### 4. Utilisation de l'API de substitution avec vecteurs de bits

Nous allons implémenter une classe utilisant l'API de substitution. Cette approche peut améliorer les performances en réutilisant les contraintes génériques et en substituant uniquement les valeurs spécifiques à la grille de Sudoku en cours de résolution.


In [9]:
using Microsoft.Z3;


public class Z3BitVectorSolverSubstitution : Z3BitVectorSolverBase
{
    public override SudokuGrid Solve(SudokuGrid s)
    {
        SudokuGrid solution = new SudokuGrid();
        SudokuSolve(s, ref solution);
        return solution;
    }

    public void SudokuSolve(SudokuGrid grid, ref SudokuGrid solution)
    {
        var substExprs = new List<Expr>();
        var substVals = new List<Expr>();

        for (int i = 0; i < 9; i++)
        for (int j = 0; j < 9; j++)
            if (grid.Cells[i,j] != 0)
            {
                substExprs.Add(CellVariables[i][j]);
                substVals.Add(GetConstExpr(grid.Cells[i,j]));
            }

        // Utiliser l'API de substitution pour appliquer les contraintes spécifiques
        BoolExpr instance_c = (BoolExpr)GenericContraints.Substitute(substExprs.ToArray(), substVals.ToArray());
        Solver solver = ctx.MkSolver();
        solver.Assert(instance_c);
        if (solver.Check() == Status.SATISFIABLE)
        {
            Model m = solver.Model;
            for (uint i = 0; i < 9; i++)
            {
                for (uint j = 0; j < 9; j++)
                {
                    if (grid.Cells[i,j] == 0)
                    {
                        solution.Cells[i,j] = ((BitVecNum)m.Evaluate(CellVariables[i][j])).Int;
                    }
                    else
                    {
                        solution.Cells[i,j] = grid.Cells[i,j];
                    }
                }
            }
        }
        else
        {
            Console.WriteLine("Failed to solve sudoku");
            throw new Exception("Failed to solve sudoku");
        }
    }
}

Console.WriteLine("Classe Z3BitVectorSolverSubstitution definie (solveur Z3 avec substitution)");

Classe Z3BitVectorSolverSubstitution definie (solveur Z3 avec substitution)


### 5. Utilisation de l'API de tactiques avec vecteurs de bits

Nous allons maintenant explorer l'utilisation de l'API de tactiques de Z3 pour résoudre les Sudokus. Les tactiques permettent d'appliquer des transformations et des simplifications spécifiques pour améliorer l'efficacité de la résolution.

In [10]:
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Threading.Tasks;

internal class Z3BitSub_Tactic : Z3BitVectorSolverBase
{
    public override SudokuGrid Solve(SudokuGrid s)
    {
        SudokuGrid solution = new SudokuGrid();
        SudokuSolve(s, ref solution);
        return solution;
    }
    public void SudokuSolve(SudokuGrid grid, ref SudokuGrid solution)
    {
        var substExprs = new List<Expr>();
        var substVals = new List<Expr>();
        for (int i = 0; i < 9; i++)
            for (int j = 0; j < 9; j++)
                if (grid.Cells[i,j] != 0)
                {
                    substExprs.Add(CellVariables[i][j]);
                    substVals.Add(GetConstExpr(grid.Cells[i,j]));
                }
        BoolExpr instance_c = (BoolExpr)GenericContraints.Substitute(substExprs.ToArray(), substVals.ToArray());
        Solver solver = ctx.MkSolver();
        solver.Assert(instance_c);
        BoolExpr puzzleConstraints = GetPuzzleConstraints(grid);
        
        // Utiliser l'API de tactiques pour simplifier et résoudre le Sudoku
        Tactic tactic = ctx.MkTactic("simplify");
        Goal goal = ctx.MkGoal();
        goal.Assert(ctx.MkAnd(GenericContraints, puzzleConstraints));
        ApplyResult applyResult = tactic.Apply(goal);
        if (applyResult.NumSubgoals > 0)
        {
            Goal newGoal = applyResult.Subgoals[0];
            solver.Assert(newGoal.Formulas);
            if (solver.Check() == Status.SATISFIABLE)
            {
                Model m = solver.Model;
                for (uint i = 0; i < 9; i++)
                {
                    for (uint j = 0; j < 9; j++)
                    {
                        if (grid.Cells[i,j] == 0)
                        {
                            solution.Cells[i,j] = ((BitVecNum)m.Evaluate(CellVariables[i][j])).Int;
                        }
                        else
                        {
                            solution.Cells[i,j] = grid.Cells[i,j];
                        }
                    }
                }
            }
            else
            {
                Console.WriteLine("Failed to solve sudoku");
                throw new Exception("Failed to solve sudoku");
            }
        }
    }
}

Console.WriteLine("Classe Z3BitSub_Tactic definie (solveur Z3 avec tactiques)");

Classe Z3BitSub_Tactic definie (solveur Z3 avec tactiques)


### 6. Comparaison des solveurs

Pour comparer les différentes implémentations de solveurs, nous utiliserons une approche similaire à celle de notre notebook OR-Tools. Nous évaluerons les performances de chaque solver sur des puzzles de Sudoku de différentes difficultés (Facile, Moyen, Difficile).

## Exercice : Compter le nombre de solutions avec Z3

**Objectif :**
Utilisez Z3 pour compter le nombre total de solutions d'un puzzle en
ajoutant itérativement des contraintes d'exclusion.

**Indice :**
Après chaque solution trouvée, ajoutez une contrainte qui exclut cette solution
et relancez le solver.


In [11]:
// EXERCICE : Compter le nombre de solutions avec Z3
public int CountSolutionsWithZ3(int[,] puzzle, int maxCount = 100)
{
    // TODO: Comptez les solutions en ajoutant des contraintes d'exclusion
    // a chaque iteration
    return 0; // TODO etudiant
}


In [12]:
// Comparaison des quatre encodages Z3 sur des grilles réelles.
// Budget borne pour un temps d'exécution raisonnable : 3 grilles par difficulté, 3 s max par grille.
// Les grilles "Hard" (top95) peuvent dépasser ce budget pour certains encodages : un dépassement
// (statut "Timeout" / "Disqualified") est alors un résultat en soi, pas une erreur.
var solvers = new List<(string Name, ISudokuSolver Solver)>
{
    ("Z3 Int Solver Simple", new Z3IntSolverSimple()),
    ("Z3 Bit Vector Solver Simple", new Z3BitVectorSolverSimple()),
    ("Z3 Bit Vector Solver Substitution", new Z3BitVectorSolverSubstitution()),
    ("Z3 Bit Vector Solver Tactic", new Z3BitSub_Tactic())
};

var results = SudokuHelper.TestSolvers(solvers, numberOfSudokus: 3, timeLimitMilliseconds: 3000);

// Tableau texte des résultats (committe dans le notebook ; les graphiques Plotly ci-dessous sont interactifs).
Console.WriteLine(String.Format("{0,-36} {1,-10} {2,12} {3,8}  {4}", "Encodage", "Difficulte", "Temps (ms)", "Resolus", "Statut"));
Console.WriteLine(new string('-', 84));
foreach (var r in results)
    Console.WriteLine(String.Format("{0,-36} {1,-10} {2,12:F1} {3,8}  {4}", r.SolverName, r.Difficulty, r.Time, r.SolvedCount, r.Status));

SudokuHelper.DisplayResults(results);

Testing Z3 Bit Vector Solver Tactic on Hard sudokus...

Encodage                             Difficulte   Temps (ms)  Resolus  Statut


------------------------------------------------------------------------------------


Z3 Int Solver Simple                 Easy              314,5        3  Success


Z3 Int Solver Simple                 Medium            848,9        3  Success


...

Z3 Bit Vector Solver Substitution    Medium            267,6        3  Success


Z3 Bit Vector Solver Substitution    Hard              319,7        3  Success


Z3 Bit Vector Solver Tactic          Easy              223,4        3  Success


Z3 Bit Vector Solver Tactic          Medium            296,6        3  Success


Z3 Bit Vector Solver Tactic          Hard              331,7        3  Success


### Interprétation des résultats

La cellule précédente lance le banc d'essai comparatif (`TestSolvers` + `DisplayResults`) sur trois niveaux de difficulté, avec un budget borné (3 grilles par difficulté, 3 s max par grille) afin que la comparaison tienne dans un temps raisonnable. Le tableau texte capture les temps de résolution réels ; les graphiques Plotly, eux, restent interactifs et ne sont pas sérialisés dans le notebook.

Sur l'exécution committee, les quatre encodages résolvent l'intégralité des grilles (statut `Success`, 3/3 à chaque niveau). On observe néanmoins un contraste net : l'encodage **entier** est le plus rapide sur les grilles faciles mais se dégrade sur les grilles difficiles (`top95`), tandis que les encodages **par vecteurs de bits** restent plus stables et prennent l'avantage sur les grilles les plus dures. Les mesures exactes varient d'une exécution à l'autre (contexte partagé, charge machine), mais cette tendance qualitative est robuste.

| Solveur | Approche | Avantages | Inconvénients |
|---------|----------|-----------|---------------|
| Z3 Int Solver Simple | Entiers | Implémentation directe, facile à comprendre | Se dégrade sur les grilles les plus difficiles |
| Z3 Bit Vector Simple | Vecteurs 4 bits | Bonne performance, stable sur toutes les difficultés | Même structure de contraintes que l'int solver |
| Z3 Bit Vector Substitution | Substitution | Réutilise les contraintes génériques | Plus complexe à mettre en oeuvre |
| Z3 Bit Vector Tactic | Tactiques | Simplification automatique | Overhead potentiel |

**Points cles** :

1. **Vecteurs de bits** : representer chaque case sur 4 bits (valeurs 1-9) suffit et donne des variables plus compactes que les entiers ; c'est aussi l'encodage le plus stable sur les grilles difficiles dans le banc d'essai ci-dessus.

2. **API de substitution** : permet de réutiliser les contraintes génériques en substituant uniquement les valeurs spécifiques, évitant de redéclarer toutes les contraintes.

3. **Tactiques** : les tactiques comme "simplify" pré-traitent les contraintes, mais l'overhead ne se justifie pas toujours pour des Sudoku simples.

4. **Performance** : sur des grilles faciles les quatre encodages se valent ; l'écart se creuse sur les grilles difficiles, où les vecteurs de bits gardent l'avantage. Le budget borné du banc d'essai (statut `Timeout` / `Disqualified` possible sur `top95`) fait partie de la mesure : un dépassement est un résultat, pas une erreur.

> **Note technique** : les solveurs Z3 utilisent un contexte statique partage pour optimiser la performance. Dans une application de production, il faudrait gerer le cycle de vie du contexte plus proprement.

### Conclusion

Dans ce notebook, nous avons exploré différentes approches pour résoudre des puzzles de Sudoku en utilisant Z3. Nous avons commencé par une implémentation simple en utilisant des entiers, puis nous avons introduit des méthodes plus sophistiquées utilisant des vecteurs de bits, l'API de substitution et les tactiques de pré-traitement. Les cellules de démonstration montrent chaque encodage résolvant une grille réelle (grille de départ, grille complète, zéro erreur), et le banc d'essai borné compare leurs temps de résolution sur trois niveaux de difficulté.

Ces quatre formulations illustrent comment un même problème peut être modélisé de plusieurs façons avec Z3. À l'échelle d'un Sudoku 9x9, les quatre approches résolvent les grilles en moins d'une seconde ; l'encodage entier est le plus rapide sur les grilles faciles, mais les encodages par vecteurs de bits se révèlent plus stables sur les grilles les plus difficiles. Le choix se justifie donc à la fois par la lisibilité, la réutilisabilité des contraintes et le profil de performance visé.

Merci d'avoir suivi ce notebook, et j'espère que cela vous a permis de mieux comprendre comment utiliser Z3 pour résoudre des problèmes de contraintes complexes comme les puzzles de Sudoku.

## Exercices

### Exercice 1 : Implémenter le Sudoku X avec diagonales

Le **Sudoku X** est une variante où les deux diagonales principales doivent également contenir des chiffres distincts.

**Objectif** : Étendre `Z3BitVectorSolverBase` pour ajouter des contraintes de diagonale.

**Indices** :
1. La diagonale principale contient `CellVariables[i][i]` pour i de 0 à 8
2. L'anti-diagonale contient `CellVariables[i][8-i]` pour i de 0 à 8
3. Utiliser `ctx.MkDistinct()` comme pour les lignes et colonnes
4. `ctx` et `CellVariables` sont accessibles statiquement depuis la classe de base

**Vérification** : Vérifiez d'abord avec `EnforceDiagonals = false` (comportement identique au solveur de base).

## Exercice : Coloration de graphe avec Z3

**Objectif :**
Utilisez Z3 pour résoudre un problème de coloration de graphe : colorier
un graphe avec k couleurs telles que deux noeuds adjacents n'ont jamais la même couleur.

**Indice :**
Créez une variable entière par noeud et ajoutez des contraintes de différence pour chaque arête.


In [13]:
// EXERCICE : Coloration de graphe avec Z3
public Dictionary<int, int> SolveGraphColoring(int[,] adjacencyMatrix, int numColors)
{
    // TODO: Modelisez le probleme de coloration de graphe avec Z3
    // Retournez un dictionnaire (noeud -> couleur)
    return null; // TODO etudiant
}


In [14]:
// Exercice 1 : Sudoku X avec contraintes de diagonale (Z3 BitVector)
using Microsoft.Z3;

public class SudokuXZ3Solver : Z3BitVectorSolverBase
{
    public bool EnforceDiagonals { get; set; } = true;

    public override SudokuGrid Solve(SudokuGrid grid)
    {
        var solver = ctx.MkSolver();

        // TODO : Ajouter les contraintes generiques standards
        // solver.Assert(GenericContraints);

        if (EnforceDiagonals)
        {
            // TODO : Extraire et contraindre la diagonale principale
            // var mainDiag = new BitVecExpr[9];
            // for (int i = 0; i < 9; i++) mainDiag[i] = CellVariables[i][i];
            // solver.Assert(ctx.MkDistinct(mainDiag));

            // TODO : Extraire et contraindre l'anti-diagonale
            // var antiDiag = new BitVecExpr[9];
            // for (int i = 0; i < 9; i++) antiDiag[i] = CellVariables[i][8 - i];
            // solver.Assert(ctx.MkDistinct(antiDiag));
        }

        // TODO : Ajouter les contraintes du puzzle et résoudre
        // solver.Assert(GetPuzzleConstraints(grid));
        // if (solver.Check() == Status.SATISFIABLE) { ... extraire la solution ... }

        return grid;
    }
}

Console.WriteLine("TODO : Implementez SudokuXZ3Solver avec les contraintes de diagonale");

TODO : Implementez SudokuXZ3Solver avec les contraintes de diagonale


### Exercice 2 : Vérifier l'unicité d'une solution avec Z3

Z3 permet de chercher toutes les solutions en ajoutant progressivement des contraintes d'exclusion.

**Objectif** : Implémenter `HasUniqueSolution` qui vérifie qu'un puzzle a exactement une solution.

**Indices** :
1. Trouver la première solution S1
2. Construire la contrainte d'exclusion : "au moins une cellule diffère de S1"
   - `ctx.MkOr(ctx.MkNot(ctx.MkEq(CellVariables[i][j], valeur_S1)))` pour chaque cellule
3. Si `solver.Check()` retourne `UNSATISFIABLE` après ajout : la solution est unique
4. Technique appelée "exclusion de modèle" (model blocking)

**Vérification** : Testez sur un puzzle facile (solution unique attendue).

In [15]:
// Exercice 2 : Vérification d'unicité des solutions avec Z3
public class Z3SudokuUnicityChecker
{
    private static Context ctx = Z3IntSolverSimple.ctx;
    private static IntExpr[][] CellVariables = Z3IntSolverSimple.CellVariables;

    /// <summary>
    /// Vérifie si un puzzle a exactement une solution.
    /// Strategie : trouver S1, puis chercher S2 != S1. Si impossible -> unique.
    /// </summary>
    public bool HasUniqueSolution(SudokuGrid puzzle)
    {
        var s = ctx.MkSolver();

        // TODO : 1. Ajouter les contraintes generiques et celles du puzzle
        // s.Assert(Z3IntSolverSimple.GenericContraints);
        // Indice : voir Z3IntSolverSimple.GetPuzzleConstraints(puzzle)

        // TODO : 2. Vérifier satisfiabilité et extraire S1
        // if (s.Check() != Status.SATISFIABLE) return false;
        // Model m1 = s.Model;

        // TODO : 3. Construire la contrainte "au moins une cellule differe de S1"
        // BoolExpr[] differences = new BoolExpr[81];
        // for (int i = 0; i < 9; i++)
        //     for (int j = 0; j < 9; j++)
        //     {
        //         var val = ((IntNum)m1.Evaluate(CellVariables[i][j])).Int;
        //         differences[i * 9 + j] = ctx.MkNot(ctx.MkEq(CellVariables[i][j], ctx.MkInt(val)));
        //     }
        // s.Assert(ctx.MkOr(differences));

        // TODO : 4. Retourner true si aucune deuxième solution n'existe
        // return s.Check() == Status.UNSATISFIABLE;

        return false;
    }
}

// Test
var checker2 = new Z3SudokuUnicityChecker();
var easyPuzzle2 = SudokuHelper.GetSudokus(SudokuDifficulty.Easy).First();
// bool unique = checker2.HasUniqueSolution(easyPuzzle2);
// Console.WriteLine($"Solution unique : {unique}");
Console.WriteLine("TODO : Implementez Z3SudokuUnicityChecker.HasUniqueSolution()");

TODO : Implementez Z3SudokuUnicityChecker.HasUniqueSolution()


### Exercice 3 : Comparer les tactiques Z3

Z3 propose différentes tactiques pour pré-traiter les formules avant résolution. Chaque tactique à ses avantages selon le type de problème.

**Objectif** : Mesurer l'impact des tactiques sur les performances de résolution.

**Indices** :
1. Créer une tactique : `ctx.MkTactic("simplify")` ou `ctx.MkTactic("bit-blast")`
2. Créer un goal : `ctx.MkGoal()` puis `goal.Assert(contraintes)`
3. Appliquer : `tactic.Apply(goal)` retourne un `ApplyResult`
4. Passer les sous-goals au solveur : `solver.Assert(result.Subgoals[0].Formulas)`
5. Tactiques a tester : `"simplify"`, `"bit-blast"`, `"ctx-solver-simplify"`, `"solve-eqs"`

**Vérification** : `"bit-blast"` est généralement efficace pour les BitVectors.

In [16]:
// Exercice 3 : Comparaison des tactiques Z3
using Microsoft.Z3;
using System.Diagnostics;

var tacticPuzzle = SudokuHelper.GetSudokus(SudokuDifficulty.Medium).First();
var tacticCtx = Z3BitVectorSolverBase.ctx;
var genericConstr = Z3BitVectorSolverBase.GenericContraints;
var puzzleConstrHelper = new Z3BitVectorSolverSimple();
var puzzleConstr = puzzleConstrHelper.GetPuzzleConstraints(tacticPuzzle);

string[] tacticNames = { "simplify", "ctx-solver-simplify", "bit-blast", "solve-eqs" };

Console.WriteLine("Comparaison des tactiques Z3 :");
Console.WriteLine(new string('-', 40));

// TODO : Pour chaque tactique, créer et appliquer, mesurer le temps de résolution
// foreach (var tacticName in tacticNames)
// {
//     var tactic = tacticCtx.MkTactic(tacticName);
//     var goal = tacticCtx.MkGoal();
//     goal.Assert(tacticCtx.MkAnd(genericConstr, puzzleConstr));
//     var sw = Stopwatch.StartNew();
//     var result = tactic.Apply(goal);
//     var solver = tacticCtx.MkSolver();
//     if (result.NumSubgoals > 0)
//         solver.Assert(result.Subgoals[0].Formulas);
//     var status = solver.Check();
//     sw.Stop();
//     Console.WriteLine($"{tacticName,-25} : {sw.ElapsedMilliseconds,6} ms - {status}");
// }

Console.WriteLine("TODO : Implementez la comparaison des tactiques Z3");

Comparaison des tactiques Z3 :


----------------------------------------


TODO : Implementez la comparaison des tactiques Z3


## Exercice : Vérification de propriétés avec Z3

### Enonce

Utilisez Z3 pour vérifier des propriétés logiques sur les grilles de Sudoku. Implémentez les fonctions suivantes avec l'API Z3 en C# :

1. `HasUniqueSolution(SudokuGrid puzzle)` : Vérifie si une grille a exactement une solution
   - Trouver la première solution S1
   - Ajouter la contrainte "la solution n'est pas S1"
   - Si aucune deuxième solution -> unicité vérifiée
   
2. `IsMinimal(SudokuGrid puzzle)` : Vérifie si le puzzle est minimal (supprimer une valeur initiale crée une ambiguïté)
   - Pour chaque cellule fixe (r, c) avec valeur v :
     - Créer une grille sans cette cellule
     - Vérifier que `HasUniqueSolution` retourne `false` (plus d'une solution)

3. Testez avec le puzzle facile : vérifiez unicité et minimalité

**Indice :**

Pour `HasUniqueSolution`, après avoir trouvé S1 = {cells[i][j] = v[i][j]}, ajoutez la contrainte `Or(cell[i][j] != v[i][j])` pour toutes les cellules. Cette contrainte dit "au moins une cellule diffère de S1".

In [17]:
// Exercice : Vérification de propriétés avec Z3

public class Z3SudokuPropertyChecker
{
    private static Context ctx = Z3IntSolverSimple.ctx;

    /// <summary>
    /// Vérifie si un puzzle a exactement une solution.
    /// Strategie : trouver S1, puis chercher S2 != S1. Si impossible -> unique.
    /// </summary>
    public bool HasUniqueSolution(SudokuGrid puzzle)
    {
        // TODO : Implémenter la vérification d'unicité
        // 1. Construire les contraintes generiques + contraintes du puzzle
        // 2. Trouver la première solution S1
        // 3. Ajouter la contrainte "au moins une cellule differe de S1"
        // 4. Vérifier si une deuxième solution existe
        return false;
    }

    /// <summary>
    /// Vérifie si le puzzle est minimal :
    /// supprimer n'importe quelle valeur initiale crée une ambiguïté.
    /// </summary>
    public bool IsMinimal(SudokuGrid puzzle)
    {
        // TODO : Implémenter la vérification de minimalité
        // Pour chaque cellule fixee (i, j) :
        //   - Créer une copie du puzzle sans la valeur en (i, j)
        //   - Vérifier que HasUniqueSolution retourne false
        // Si toutes les cellules satisfont cette condition -> minimal
        return false;
    }
}

// Test : vérifier les propriétés sur le puzzle facile
var checker = new Z3SudokuPropertyChecker();
var easyPuzzle = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).First();
Console.WriteLine("Test de verification de proprietes Z3");
Console.WriteLine("Puzzle selectionne :");
Console.WriteLine(easyPuzzle);
// bool unique = checker.HasUniqueSolution(easyPuzzle);
// bool minimal = checker.IsMinimal(easyPuzzle);
// Console.WriteLine($"Solution unique : {unique}");
// Console.WriteLine($"Puzzle minimal : {minimal}");

Test de verification de proprietes Z3


Puzzle selectionne :


-------------------------------
| 4       |         | 8     5 | 
|    3    |         |         | 
|         | 7       |         | 
-------------------------------
|    2    |         |    6    | 
|         |    8    | 4       | 
|         |    1    |         | 
-------------------------------
|         | 6     3 |    7    | 
| 5       | 2       |         | 
| 1     4 |         |         | 
-------------------------------


### A retenir

Z3 offre **quatre formulations** de résolution Sudoku, de la plus simple à la plus optimisée :

| Formulation | Variables | Avantage |
|-------------|-----------|----------|
| **IntExpr simple** | 81 IntVar | Modélisation directe, facile à comprendre |
| **BitVec 4 bits** | 81 BitVec | Espace reduit (4 bits vs 32 bits) |
| **Substitution API** | 81 BitVec | Réutilisation du modèle générique |
| **Tactiques** | 81 BitVec | Pre-traitement (simplify, bit-blast) |

**Lecons** :
1. L'API **Substitution** permet de séparer le modèle générique (contraintes Sudoku) des valeurs spécifiques au puzzle.
2. Les **tactiques** (simplify, bit-blast, ctx-solver-simplify) pré-traitent les formules pour accélérer la résolution.
3. L'**exclusion de modèle** (model blocking) vérifie l'unicité de la solution en ajoutant des contraintes d'exclusion.
4. Z3 reste l'outil de référence pour la **vérification formelle** de propriétés (unicité, minimalité).


---

**Navigation** : [<< Sudoku-11 Choco C#](Sudoku-11-Choco-Csharp.ipynb) | [Index](README.md) | [Sudoku-13 Symbolic Automata C# >>](Sudoku-13-SymbolicAutomata-Csharp.ipynb)

**Voir aussi** : 
- [CSP-3-Avance](../Search/Part2-CSP/CSP-3-Advanced.ipynb) - Contraintes globales avancees
- [SymbolicAI](../SymbolicAI/README.md) - Série sur l'IA symbolique et Z3